# Football Transfer Fee Prediction Pipeline

This notebook implements a clean, production-oriented version of the transfer fee prediction model. It includes data extraction, sophisticated feature engineering (including FIFA ratings integration), and model evaluation using both Linear Regression and Random Forest Regressor.

## 1. Environment Setup and Data Loading

In [1]:
import numpy as np
import pandas as pd
import os
import zipfile
import unicodedata
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from fuzzywuzzy import process

# Constants
TRANSFERS_PATH = '../data/transfers.zip'
RATINGS_PATH = '../data/ratings.zip'

def normalize_text(s):
    if not isinstance(s, str): return s
    return unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('utf-8')

transfers = pd.read_csv(TRANSFERS_PATH, compression='zip')
transfers.head()

## 2. Initial Data Cleaning

We clean the seasons, normalize positions, and remove inconsistent rows (e.g., Age 0).

In [2]:
# Season to int
transfers['Season_transferred'] = transfers['Season'].str.split('-').str[0].astype('int64')
transfers = transfers.drop(columns=['Season'])

# Position mapping
transfers.Position = transfers.Position.replace(
    to_replace=['Second Striker', 'Centre-Forward', 'Sweeper'],
    value=['Forward', 'Forward', 'Defender']
)

# Drop inconsistent positions and Age 0
transfers_cleaned = transfers[~((transfers.Position == 'Midfielder') | (transfers.Position == 'Defender') | (transfers.Age == 0))].copy()

# Scale currency to millions
transfers_cleaned['Transfer_fee_in_mln'] = transfers_cleaned['Transfer_fee'] / 1e6
transfers_cleaned['Market_value_in_mln'] = transfers_cleaned['Market_value'] / 1e6
transfers_cleaned = transfers_cleaned.drop(columns=['Transfer_fee', 'Market_value'])

transfers_cleaned.head()

## 3. Feature Engineering: Basic Features

Extracting Lastnames and normalizing text for reliable merging.

In [3]:
transfers_2015_18 = transfers_cleaned[transfers_cleaned['Season_transferred'] > 2014].copy()
transfers_2015_18['Lastname'] = transfers_2015_18['Name'].apply(lambda x: x.split(' ')[-1])

for col in ['Lastname', 'Team_from', 'Team_to']:
    transfers_2015_18[col] = transfers_2015_18[col].apply(normalize_text)

transfers_2015_18.head()

## 4. Advanced Feature Engineering: FIFA Ratings Integration

We load FIFA player ratings (Overall/Potential) and use multiple fallback strategies to recover missing metadata like Nationality.

In [4]:
nat_frames = []
with zipfile.ZipFile(RATINGS_PATH) as zf:
    for yr in [2015, 2016, 2017, 2018]:
        f = pd.read_csv(zf.open(f'players_{yr-2000}.csv'))
        nat_col = 'nationality_name' if 'nationality_name' in f.columns else ('nationality' if 'nationality' in f.columns else None)
        if nat_col is None: continue

        mini = f[['short_name', 'club', nat_col, 'overall', 'potential']].copy()
        mini['Lastname'] = mini['short_name'].str.split(' ').str[-1].apply(normalize_text)
        mini['club'] = mini['club'].apply(normalize_text)
        mini['Season_transferred'] = yr
        mini = mini.rename(columns={nat_col: 'Nationality'})
        nat_frames.append(mini)

fifa_nat = pd.concat(nat_frames, ignore_index=True)
fifa_nat['Nationality'] = fifa_nat['Nationality'].apply(normalize_text)

fifa_nat_best_club = fifa_nat.sort_values('overall', ascending=False).drop_duplicates(subset=['Lastname', 'club', 'Season_transferred'])
fifa_nat_best_name_season = fifa_nat.sort_values('overall', ascending=False).drop_duplicates(subset=['Lastname', 'Season_transferred'])

print(f'FIFA lookup rows (club-level): {len(fifa_nat_best_club)}')
print(f'FIFA lookup rows (lastname-season): {len(fifa_nat_best_name_season)}')

FIFA lookup rows (club-level): 63797
FIFA lookup rows (lastname-season): 46302


## 5. Dataset Merging and Fallback Recovery

We attempt to match transfers to FIFA data using Team_from, then Team_to, and finally a broader Lastname+Season search.

In [5]:
# Merge on Team_from
transfers_fe = transfers_2015_18.merge(
    fifa_nat_best_club.rename(columns={'club': 'Team_from', 'overall': 'overall_from', 'potential': 'potential_from'}),
    on=['Lastname', 'Team_from', 'Season_transferred'], how='left'
)

# Fallback on Team_to for nulls
null_mask = transfers_fe['overall_from'].isna()
to_lookup = fifa_nat_best_club.rename(columns={'club': 'Team_to', 'overall': 'overall_to', 'potential': 'potential_to', 'Nationality': 'Nat_to'})
transfers_fe = transfers_fe.merge(to_lookup, on=['Lastname', 'Team_to', 'Season_transferred'], how='left')

# Final broader fallback
transfers_fe['overall'] = transfers_fe['overall_from'].combine_first(transfers_fe['overall_to'])
transfers_fe['potential'] = transfers_fe['potential_from'].combine_first(transfers_fe['potential_to'])

final_df = transfers_fe.dropna(subset=['overall']).copy()
print(f"Nationality nulls after fallback: {final_df['Nationality'].isna().sum()} / {len(final_df)}")

Nationality nulls after fallback: 2 / 731


## 6. Engineered Indicators for Bias and Performance

Creating buckets for Age and Position, and purchasing power indicators for leagues and clubs.

In [6]:
#Purchasing power proxy
league_median = final_df.groupby('League_to')['Transfer_fee_in_mln'].median()
final_df['league_median_fee_to'] = final_df['League_to'].map(league_median)

# Age buckets
final_df['age_bucket'] = pd.cut(final_df['Age'], bins=[14, 20, 24, 27, 30, 50], labels=['<20', '20-23', '24-26', '27-29', '30+'])

# Position buckets
pos_map = {
    'Centre-Back': 'Defender', 'Right-Back': 'Defender', 'Left-Back': 'Defender',
    'Central Midfield': 'Midfielder', 'Attacking Midfield': 'Midfielder', 'Defensive Midfield': 'Midfielder',
    'Right Winger': 'Winger', 'Left Winger': 'Winger', 'Forward': 'Forward', 'Goalkeeper': 'Goalkeeper'
}
final_df['pos_group'] = final_df['Position'].map(pos_map).fillna('Other')

final_df.head()

## 7. Model Training: Linear Regression Baseline

In [7]:
# Drop name-specific identifiers
df_ml = final_df.drop(columns=['Name', 'Lastname', 'Team_from', 'Team_to', 'Nationality', 'overall_from', 'potential_from', 'overall_to', 'potential_to', 'Nat_to'])
df_ml = pd.get_dummies(df_ml)

X = df_ml.drop(columns=['Transfer_fee_in_mln'])
y = df_ml['Transfer_fee_in_mln']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=69)

lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)

print(f"MSE: {mean_squared_error(y_test, y_pred):.2f}")
print(f"R2: {r2_score(y_test, y_pred):.2f}")

MSE: 74.64
R2: 0.21


## 8. Improved Modeling: Random Forest Regressor

Using an ensemble method to capture non-linear relationships between ability, market value, and purchasing power.

In [8]:
rfr = RandomForestRegressor(n_estimators=270, max_features=650, random_state=169)
rfr.fit(X_train, y_train)
y_pred_rfr = rfr.predict(X_test)

print(f"MSE: {mean_squared_error(y_test, y_pred_rfr):.2f}")
print(f"R2: {r2_score(y_test, y_pred_rfr):.2f}")

MSE: 47.90
R2: 0.80
